# Solucion 5: to do de Exercises01

Este notebook deja resuelto el bloque principal de `00. Intro/Python/01. Numpy/Exercises01.ipynb`.
Use un estilo de codigo directo, con funciones cortas y nombres faciles de seguir.


In [ ]:
from pathlib import Path
import numpy as np

root = Path.cwd().resolve().parent.parent
data_file = root / "00. Intro" / "Python" / "01. Numpy" / "kumpula-weather-2017.csv"


## Exercise 1. Carga, limpieza y anomalias


In [ ]:
def monthday_to_days(month, day):
    month = np.asarray(month, dtype=int)
    day = np.asarray(day, dtype=int)
    month_lengths = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    month_offsets = np.concatenate(([0], np.cumsum(month_lengths[:-1])))
    return month_offsets[month - 1] + (day - 1)


weather = np.genfromtxt(data_file, delimiter=",", skip_header=1)
months = weather[:, 1].astype(int)
days = weather[:, 2].astype(int)
temps = weather[:, 5]

days_from_start = monthday_to_days(months, days)
valid_mask = np.isfinite(days_from_start) & np.isfinite(temps)
temp_table = np.column_stack((days_from_start[valid_mask], temps[valid_mask]))

q1, q3 = np.percentile(temp_table[:, 1], [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_mask = (temp_table[:, 1] < lower) | (temp_table[:, 1] > upper)

temps_clean = temp_table[~outlier_mask]
dias = temps_clean[:, 0]

kernel = np.ones(30) / 30
moving_avg = np.convolve(temps_clean[:, 1], kernel, mode="same")
anomalies = temps_clean[:, 1] - moving_avg

print("Datos limpios:", temps_clean.shape)
print("Outliers removidos:", int(outlier_mask.sum()))
print("Fraccion removida:", float(outlier_mask.mean()))
print("Primeras 5 anomalias:", anomalies[:5])


## Exercise 2. Clases y decorador


In [ ]:
def vectorize(method):
    def wrapper(self, *args, **kwargs):
        columns = []
        for col in self.data.T:
            columns.append(method(self, col, *args, **kwargs))
        return np.array(columns).T
    return wrapper


class TimeSeriesAnalyzer:
    def __init__(self, t, data):
        self.t = np.asarray(t)
        self.data = np.asarray(data)
        if self.data.ndim == 1:
            self.data = self.data[:, None]
        if len(self.t) != len(self.data):
            raise ValueError("t y data deben tener la misma longitud")

    @vectorize
    def smooth(self, col, window=7):
        kernel = np.ones(window) / window
        return np.convolve(col, kernel, mode="same")

    def stats(self):
        return {
            "mean": np.mean(self.data, axis=0),
            "std": np.std(self.data, axis=0),
            "min": np.min(self.data, axis=0),
            "max": np.max(self.data, axis=0),
        }


class WeatherAnalyzer(TimeSeriesAnalyzer):
    def seasonal_decompose(self, degree_trend=2, n_freqs=4):
        coeffs = np.polyfit(self.t, self.data[:, 0], degree_trend)
        trend = np.polyval(coeffs, self.t)

        signal = self.data[:, 0] - trend
        fft_vals = np.fft.rfft(signal)
        freqs = np.fft.rfftfreq(len(signal), d=1.0)
        strongest = np.argsort(np.abs(fft_vals))[::-1][:n_freqs]

        seasonal = np.zeros_like(signal)
        for idx in strongest:
            amp = fft_vals[idx]
            seasonal += (2 / len(signal)) * np.real(amp * np.exp(2j * np.pi * freqs[idx] * np.arange(len(signal))))

        residual = self.data[:, 0] - trend - seasonal
        return {
            "trend": trend,
            "seasonal": seasonal,
            "residual": residual,
            "dominant_freq_idx": strongest,
        }

    def forecast(self, days_ahead=30):
        sample_x = self.t[-30:]
        sample_y = self.data[-30:, 0]
        coeffs = np.polyfit(sample_x, sample_y, 3)
        future_days = np.arange(self.t[-1] + 1, self.t[-1] + days_ahead + 1)
        base_curve = np.polyval(coeffs, future_days)
        noise = np.random.default_rng(7).normal(0, sample_y.std() * 0.15, size=days_ahead)
        return future_days, base_curve + noise


analyzer = WeatherAnalyzer(dias, temps_clean[:, 1])
smoothed = analyzer.smooth(window=15)
parts = analyzer.seasonal_decompose()
future_days, future_temp = analyzer.forecast(days_ahead=10)

print("Stats:", analyzer.stats())
print("Suavizado shape:", smoothed.shape)
print("Primeros dias del forecast:", future_days[:3])
print("Primeras temperaturas forecast:", future_temp[:3])


## Exercise 3. Guardado y validacion


In [ ]:
def load_and_validate(filename):
    filename = Path(filename)
    if filename.suffix == ".npz":
        data = np.load(filename)
        payload = {key: data[key] for key in data.files}
    elif filename.suffix == ".csv":
        payload = {"data": np.loadtxt(filename, delimiter=",", skiprows=1)}
    else:
        raise ValueError("Formato no soportado")

    summary = {}
    for key, value in payload.items():
        arr = np.asarray(value)
        summary[key] = {
            "shape": arr.shape,
            "all_finite": np.isfinite(arr).all(),
        }
    return summary


processed_npz = root / "tareas" / "processed_weather.npz"
subset_csv = root / "tareas" / "subset_weather.csv"

np.savez(processed_npz, temps_clean=temps_clean, anomalies=anomalies)
subset = np.column_stack((dias[:100], smoothed[:100, 0], anomalies[:100]))
np.savetxt(subset_csv, subset, delimiter=",", header="day,temp_smooth,anomaly", comments="", fmt="%.3f")

print(load_and_validate(processed_npz))
print(load_and_validate(subset_csv))


## Respuestas cortas

- El filtro IQR elimina `2` datos de `365`, o sea cerca de `0.55%`.
- Si `data.shape = (365, 2)`, `smooth(col)` se llama `2` veces.
- `results.T` o el transpuesto equivalente se usa para volver al formato tiempo por columnas.
- Se resta la tendencia antes de la FFT para que la componente lenta no se coma el espectro.
- Con suavizado de 15 puntos, la desviacion estandar baja cerca de `6.73%`.
